In [1]:
import os
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, accuracy_score
from peft import get_peft_model, LoraConfig, TaskType

/Users/foongming/.pyenv/versions/3.12.3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# # 1. Device setup
# if torch.backends.mps.is_available():
#     device = torch.device("mps")
# elif torch.cuda.is_available():
#     device = torch.device("cuda")
# else:
#     device = torch.device("cpu")
# print("Using device:", device)

#force CPU because Lora needs cpu 
device = torch.device("cpu")
print("Using device:", device)

# 2. Load data
train = pd.read_csv('train_preprocessed.csv')
test = pd.read_csv('test_preprocessed.csv')

train['cleaned_text'] = train['cleaned_text'].astype(str)
test['cleaned_text'] = test['cleaned_text'].astype(str)

# 3. Encode labels
y_train = LabelEncoder().fit_transform(train['Bias'])
y_test = LabelEncoder().fit_transform(test['Bias'])
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(train['Bias'])
y_test = label_encoder.transform(test['Bias'])
num_classes = len(label_encoder.classes_)


Using device: cpu


In [3]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1779 entries, 0 to 1778
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Title         1779 non-null   object
 1   cleaned_text  1779 non-null   object
 2   Bias          1779 non-null   object
dtypes: object(3)
memory usage: 41.8+ KB


In [4]:
# 4. Tokeniser
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [5]:
class NewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = int(self.labels[idx])
        encodings = self.tokenizer(
            text,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors="pt"
        )
        return {
            "input_ids": encodings["input_ids"].squeeze(0),
            "attention_mask": encodings["attention_mask"].squeeze(0),
            "labels": torch.tensor(label)
        }

In [6]:
train_dataset = NewsDataset(train['cleaned_text'], y_train, tokenizer)
test_dataset = NewsDataset(test['cleaned_text'], y_test, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=8)

In [7]:
base_model = AutoModel.from_pretrained(
    model_id,
    torch_dtype=torch.float32,
    device_map=None,
    trust_remote_code=True
)

In [8]:
# disable bnb fallback 
import peft.tuners.lora
peft.tuners.lora.bnb_available = False

In [9]:
# 7. Add LoRA adapter
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none"
)
model = get_peft_model(base_model, peft_config)
model.config.num_labels = num_classes
model.print_trainable_parameters()


'NoneType' object has no attribute 'cadam32bit_grad_fp32'
trainable params: 1,126,400 || all params: 1,035,638,784 || trainable%: 0.1088


/Users/foongming/.pyenv/versions/3.12.3/lib/python3.12/site-packages/bitsandbytes/cextension.py:34: UserWarning: The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers, 8-bit multiplication, and GPU quantization are unavailable.
  warn("The installed version of bitsandbytes was compiled without GPU support. "


In [10]:
# 8. Add classification head
model.classifier = nn.Linear(model.config.hidden_size, num_classes)
model.classifier.to(device)

Linear(in_features=2048, out_features=5, bias=True)

In [11]:
# 9. Weighted loss for imbalance
class_weights = compute_class_weight(class_weight="balanced", classes=np.unique(y_train), y=y_train)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

In [12]:
# 10. Optimizer
optimizer = torch.optim.Adam(list(model.parameters()) + list(model.classifier.parameters()), lr=5e-4)


/Users/foongming/.pyenv/versions/3.12.3/lib/python3.12/site-packages/torch/_compile.py:32: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


In [13]:
model.train()
for epoch in range(5):
    total_loss = 0
    preds_epoch, labels_epoch = [], []

    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        base_out = model.base_model(input_ids=input_ids, attention_mask=attention_mask)
        cls_embeddings = base_out.last_hidden_state[:, 0, :]
        logits = model.classifier(cls_embeddings)

        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        total_loss += loss.item()
        preds_epoch.extend(torch.argmax(logits, dim=1).cpu().numpy())
        labels_epoch.extend(labels.cpu().numpy())

    acc = accuracy_score(labels_epoch, preds_epoch)
    print(f"Epoch {epoch+1}: Loss = {total_loss / len(train_loader):.4f}, Accuracy = {acc:.4f}")



Epoch 1: Loss = 1.7228, Accuracy = 0.2248
Epoch 2: Loss = 1.6459, Accuracy = 0.2619
Epoch 3: Loss = 1.6493, Accuracy = 0.2456
Epoch 4: Loss = 1.6454, Accuracy = 0.2243
Epoch 5: Loss = 1.6544, Accuracy = 0.2614


In [14]:
# 12. Evaluation
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        base_out = model.base_model(input_ids=input_ids, attention_mask=attention_mask)
        cls_embeddings = base_out.last_hidden_state[:, 0, :]
        logits = model.classifier(cls_embeddings)

        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print(classification_report(all_labels, all_preds, target_names=label_encoder.classes_))


              precision    recall  f1-score   support

      center       0.00      0.00      0.00        40
   lean left       0.00      0.00      0.00        61
  lean right       0.08      1.00      0.15        37
        left       0.00      0.00      0.00       230
       right       0.00      0.00      0.00        77

    accuracy                           0.08       445
   macro avg       0.02      0.20      0.03       445
weighted avg       0.01      0.08      0.01       445



/Users/foongming/.pyenv/versions/3.12.3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/foongming/.pyenv/versions/3.12.3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/foongming/.pyenv/versions/3.12.3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{m